# byteSmart Ultralow Temperature: Interpretable ML Analysis

This notebook turns the Jian Sun ultralow-temperature distribution dataset into a clear, reproducible demonstration of interpretable machine learning. It is an educational investigation tool only: it identifies patterns worth reviewing and **does not decide vaccine usability**.

## Notebook map

1. Load and inspect the dataset already in Drive.
2. Visualize dry-ice loss and representative temperature behavior.
3. Train two linear-regression models.
4. Train two logistic-regression models that predict the *next* window, avoiding target leakage.
5. Interpret graphs, metrics, and limitations.

In [ ]:
# Imports and visual style. Run this cell first.
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', context='notebook')
COLORS = {'blue':'#2563EB', 'teal':'#0F766E', 'orange':'#EA580C', 'red':'#DC2626', 'gray':'#475569'}
plt.rcParams['figure.figsize'] = (11, 5)
drive.mount('/content/drive')

## 1. Find, load, and inspect the dataset

The code searches your Google Drive for the existing dataset ZIP, extracts the CSV files, and prints their dimensions. No sensor type is assumed; temperature channels are selected from the available numeric columns.

In [ ]:
# Locate the exact ZIP already stored in byteSmart Drive.
drive_root = Path('/content/drive/MyDrive')
zip_candidates = list(drive_root.rglob('14888121*.zip'))
if not zip_candidates:
    raise FileNotFoundError('Dataset ZIP not found in Google Drive.')
zip_path = zip_candidates[0]
extract_dir = Path('/content/bytesmart_ultralow_data')
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)
data_dir = next(extract_dir.rglob('14888121'))

dry_raw = pd.read_csv(data_dir / 'Test1_DryIceWeight.csv', header=None)
dry = pd.DataFrame({'hours': pd.to_numeric(dry_raw.iloc[4:, 2], errors='coerce'), 'baseline_lb': pd.to_numeric(dry_raw.iloc[4:, 5], errors='coerce'), 'refrigerated_lb': pd.to_numeric(dry_raw.iloc[4:, 6], errors='coerce')}).dropna()

test1 = pd.read_csv(data_dir / 'Test1_TempCO2O2.csv', low_memory=False).iloc[2:].copy()
test1['hours'] = pd.to_timedelta(test1['Time Elapsed'].astype(str), errors='coerce').dt.total_seconds() / 3600
test2 = pd.read_csv(data_dir / 'Test2_TempCO2O2.csv', low_memory=False).iloc[1:].copy()
test2['timestamp'] = pd.to_datetime(test2['TIMESTAMP'], errors='coerce')
test2['hours'] = (test2['timestamp'] - test2['timestamp'].min()).dt.total_seconds() / 3600

def make_numeric(frame, excluded):
    frame = frame.copy()
    for column in frame.columns:
        if column not in excluded:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame.dropna(subset=['hours'])

test1 = make_numeric(test1, {'date', 'time', 'Time Elapsed', 'hours'})
test2 = make_numeric(test2, {'TIMESTAMP', 'timestamp', 'hours'})
print(f'Loaded {len(dry):,} dry-ice observations, {len(test1):,} Test 1 rows, and {len(test2):,} Test 2 rows.')

In [ ]:
# Select temperature channels generically, excluding time and environmental gas fields.
def find_temperature_columns(frame, excluded):
    return [c for c in frame.columns if c not in excluded and pd.api.types.is_numeric_dtype(frame[c]) and frame[c].notna().sum() > 100]

temp_cols_1 = find_temperature_columns(test1, {'hours', 'O2', 'CO2', 'Ambient', 'Unnamed: 63', 'Unnamed: 64'})
temp_cols_2 = find_temperature_columns(test2, {'hours', 'O2', 'CO2'})
overview = pd.DataFrame({'File':['Dry-ice weight', 'Temperature Test 1', 'Temperature Test 2'], 'Rows':[len(dry), len(test1), len(test2)], 'Temperature channels':['Not applicable', len(temp_cols_1), len(temp_cols_2)]})
display(overview)
print('A temperature channel is a numeric, sufficiently complete column after non-temperature fields are excluded.')

## 2. What does the raw data look like?

Start here before interpreting any model. The first chart shows dry-ice mass over time. The second shows the average temperature across all available channels; the translucent band shows how much the channels disagree at each moment.

In [ ]:
# Visual 1: dry-ice mass in each observed condition.
fig, ax = plt.subplots()
ax.plot(dry['hours'], dry['baseline_lb'], marker='o', ms=3, lw=2, color=COLORS['orange'], label='Baseline')
ax.plot(dry['hours'], dry['refrigerated_lb'], marker='o', ms=3, lw=2, color=COLORS['blue'], label='Refrigerated')
ax.set(title='Dry-Ice Mass Over Time', xlabel='Elapsed time (hours)', ylabel='Measured dry-ice mass (lb)')
ax.legend(title='Condition'); sns.despine(); plt.show()

# Visual 2: an overview of the first 24 hours so the graph stays readable.
def temperature_summary(frame, columns):
    out = frame[['hours']].copy()
    out['mean_temp'] = frame[columns].mean(axis=1)
    out['min_temp'] = frame[columns].min(axis=1)
    out['max_temp'] = frame[columns].max(axis=1)
    return out.dropna()

summary1 = temperature_summary(test1, temp_cols_1)
fig, ax = plt.subplots()
shown = summary1[summary1['hours'] <= 24]
ax.plot(shown['hours'], shown['mean_temp'], color=COLORS['teal'], lw=1.5, label='Mean across temperature channels')
ax.fill_between(shown['hours'], shown['min_temp'], shown['max_temp'], color=COLORS['teal'], alpha=.18, label='Range across channels')
ax.set(title='Temperature Behavior in Test 1 (First 24 Hours)', xlabel='Elapsed time (hours)', ylabel='Recorded temperature (dataset units)')
ax.legend(); sns.despine(); plt.show()

## 3. Linear regression: estimating dry-ice mass

**Research question:** Can dry-ice mass be predicted from elapsed time and storage condition?

The diagonal line in the next chart is perfect prediction. Points closer to that line are better predicted. MAE is the average prediction error in pounds; R-squared shows how much of the variation in mass is explained by the model.

In [ ]:
# Combine the two mass series and give the model an explicit condition indicator.
dry_long = pd.concat([dry[['hours','baseline_lb']].rename(columns={'baseline_lb':'mass_lb'}).assign(condition='Baseline'), dry[['hours','refrigerated_lb']].rename(columns={'refrigerated_lb':'mass_lb'}).assign(condition='Refrigerated')], ignore_index=True)
dry_long['is_refrigerated'] = (dry_long['condition'] == 'Refrigerated').astype(int)
X = dry_long[['hours', 'is_refrigerated']]; y = dry_long['mass_lb']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)
dry_model = LinearRegression().fit(X_train, y_train)
pred = dry_model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, pred):.2f} lb | RMSE: {mean_squared_error(y_test, pred)**.5:.2f} lb | R-squared: {r2_score(y_test, pred):.3f}')
fig, ax = plt.subplots()
ax.scatter(y_test, pred, color=COLORS['blue'], alpha=.8, s=55)
limits = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(limits, limits, '--', color=COLORS['gray'], label='Perfect prediction')
ax.set(xlim=limits, ylim=limits, title='Linear Regression: Observed vs Predicted Dry-Ice Mass', xlabel='Observed mass (lb)', ylabel='Predicted mass (lb)')
ax.legend(); sns.despine(); plt.show()

## 4. Create forward-looking time windows

Raw readings are condensed into 60-reading windows. Features describe the **current** window; labels describe the **next** window. This sequence matters: it prevents the model from using the answer while making its prediction.

- `investigation_needed`: the next window has unusually rapid temperature movement (top 10% within this dataset).
- `warm_relative_window`: the next window is among the warmest 10% observed in this dataset.

These are transparent research labels, not clinical thresholds.

In [ ]:
def build_windows(frame, temp_columns, name, window_size=60):
    work = frame[['hours', 'O2', 'CO2'] + temp_columns].copy()
    work['mean_temp'] = work[temp_columns].mean(axis=1)
    work['temp_spread'] = work[temp_columns].max(axis=1) - work[temp_columns].min(axis=1)
    work = work.dropna(subset=['mean_temp', 'temp_spread', 'O2', 'CO2'])
    rows = []
    for start in range(0, len(work)-window_size+1, window_size):
        chunk = work.iloc[start:start+window_size]
        h, t = chunk['hours'].to_numpy(), chunk['mean_temp'].to_numpy()
        rows.append({'test':name, 'mean_temp':t.mean(), 'temp_spread':chunk['temp_spread'].mean(), 'O2_mean':chunk['O2'].mean(), 'CO2_mean':chunk['CO2'].mean(), 'temp_slope_per_hour':np.polyfit(h,t,1)[0] if np.ptp(h)>0 else 0})
    return pd.DataFrame(rows)

windows = pd.concat([build_windows(test1,temp_cols_1,'Test 1'), build_windows(test2,temp_cols_2,'Test 2')], ignore_index=True).dropna()
# Future labels: shift within each test so current-window values never contain the future answer.
windows['future_mean_temp'] = windows.groupby('test')['mean_temp'].shift(-1)
windows['future_temp_change'] = windows.groupby('test')['mean_temp'].shift(-1) - windows['mean_temp']
windows['future_slope'] = windows.groupby('test')['temp_slope_per_hour'].shift(-1)
windows = windows.dropna().reset_index(drop=True)
windows['investigation_needed'] = (windows['future_slope'].abs() >= windows['future_slope'].abs().quantile(.90)).astype(int)
windows['warm_relative_window'] = (windows['future_mean_temp'] >= windows['future_mean_temp'].quantile(.90)).astype(int)
features = ['mean_temp','temp_spread','O2_mean','CO2_mean','temp_slope_per_hour']
print(f'Built {len(windows):,} time windows. Investigation-needed rate: {windows.investigation_needed.mean():.1%}; warm-relative rate: {windows.warm_relative_window.mean():.1%}.')
fig, ax = plt.subplots()
sns.countplot(data=windows.melt(value_vars=['investigation_needed','warm_relative_window'], var_name='Label', value_name='Flag'), x='Label', hue='Flag', palette=[COLORS['blue'], COLORS['red']], ax=ax)
ax.set(title='Class Balance for Research Labels', xlabel='', ylabel='Number of windows'); ax.legend(title='Flag', labels=['No','Yes']); sns.despine(); plt.show()

## 5. Linear regression: predicting the next temperature change

**Research question:** Can the next-window temperature change be estimated from the current environmental state?

A positive value means the next window is warmer on average; a negative value means it is cooler. The coefficient chart uses standardized features, so longer bars indicate a stronger relationship on the same scale.

In [ ]:
X = windows[features]; y = windows['future_temp_change']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.25,random_state=42)
change_model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train,y_train)
pred = change_model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test,pred):.4f} | R-squared: {r2_score(y_test,pred):.3f}')
fig, axes = plt.subplots(1,2,figsize=(15,5))
axes[0].scatter(y_test,pred,color=COLORS['teal'],alpha=.65)
limits=[min(y_test.min(),pred.min()),max(y_test.max(),pred.max())]; axes[0].plot(limits,limits,'--',color=COLORS['gray'])
axes[0].set(title='Next-Window Change: Observed vs Predicted',xlabel='Observed change',ylabel='Predicted change',xlim=limits,ylim=limits)
coef = pd.Series(change_model.named_steps['linearregression'].coef_,index=features).sort_values()
coef.plot.barh(ax=axes[1],color=[COLORS['orange'] if x<0 else COLORS['blue'] for x in coef])
axes[1].set(title='Which Current Features Matter Most?',xlabel='Standardized coefficient',ylabel=''); sns.despine(); plt.tight_layout(); plt.show()

## 6. Logistic regression: flagging the next investigation-needed window

**Research question:** Can the current window predict whether the next window will show unusually rapid temperature movement?

Read the confusion matrix by rows: each row is the actual class, each column is the model prediction. Because investigation windows are uncommon, the model uses balanced class weights so it does not simply predict `stable` every time.

In [ ]:
X = windows[features]; y = windows['investigation_needed']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
investigation_model = make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,class_weight='balanced')).fit(X_train,y_train)
pred = investigation_model.predict(X_test); prob = investigation_model.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred,target_names=['Stable','Investigation needed']))
print(f'ROC-AUC: {roc_auc_score(y_test,prob):.3f}')
fig, axes = plt.subplots(1,2,figsize=(14,5))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Stable','Investigation needed'],colorbar=False,ax=axes[0],cmap='Blues')
axes[0].set_title('Confusion Matrix')
plot_data=pd.DataFrame({'Predicted probability':prob,'Actual label':np.where(y_test==1,'Investigation needed','Stable')})
sns.boxplot(data=plot_data,x='Actual label',y='Predicted probability',palette=[COLORS['blue'],COLORS['red']],ax=axes[1])
axes[1].axhline(.5,ls='--',color=COLORS['gray'],label='Default cutoff'); axes[1].set(title='Are Investigation Windows Given Higher Probability?',ylim=(0,1)); axes[1].legend(); sns.despine(); plt.tight_layout(); plt.show()

## 7. Logistic regression: estimating a future warm-relative probability

**Research question:** Can the current window estimate whether the *next* window will fall among the warmest 10% of the observed dataset?

ROC-AUC is 0.5 for random ranking and approaches 1.0 for perfect ranking. This graph is a ranked watchlist: the first rows are the windows the model believes most likely to be followed by a warm-relative window.

In [ ]:
X = windows[features]; y = windows['warm_relative_window']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
warm_model = make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,class_weight='balanced')).fit(X_train,y_train)
prob = warm_model.predict_proba(X_test)[:,1]
print(f'ROC-AUC: {roc_auc_score(y_test,prob):.3f}')
watchlist = X_test.copy(); watchlist['predicted_probability']=prob; watchlist['actual_next_window_warm']=y_test
display(watchlist.sort_values('predicted_probability',ascending=False).head(10).style.format({'predicted_probability':'{:.1%}'}))
fig, ax = plt.subplots()
ax.hist(prob[y_test.to_numpy()==0],bins=15,alpha=.65,color=COLORS['blue'],label='Next window not warm-relative')
ax.hist(prob[y_test.to_numpy()==1],bins=15,alpha=.65,color=COLORS['red'],label='Next window warm-relative')
ax.set(title='Predicted Probability Distribution',xlabel='Predicted probability of future warm-relative window',ylabel='Number of windows'); ax.legend(); sns.despine(); plt.show()

## 8. How to interpret and present the results

- **Use the data overview and raw plots first.** They show what the experiment actually measured before a model summarizes it.
- **Use the two linear models for estimates.** Report MAE and R-squared together; neither metric alone tells the whole story.
- **Use logistic probabilities as a prioritized review list.** A high probability means `review first`, not `unsafe`.
- **Be explicit about limits.** Labels here are based on the dataset distribution. A real deployment needs validated product requirements, calibrated instruments, documented event labels, and organization-specific escalation procedures.

This is a strong lab conversation starter because it demonstrates transparent data handling, interpretable baseline models, and a clear boundary between investigation support and product decisions.